## ✅ Part 1: Incremental Load for Weather Fact Table

In [7]:
from datetime import datetime, timedelta
from pyspark.sql.functions import col, lit, min as spark_min, max as spark_max
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import uuid

# -----------------------------------------------
# 📆 Step 0: Initialise Run Variables
# -----------------------------------------------

run_id = str(uuid.uuid4())                           # Unique run ID
start_time = datetime.utcnow()                       # Start time
rows_before, rows_inserted, rows_after = 0, 0, 0     # Row counters
error_count = 0                                      # Error tracking
status = "success"                                   # Default status

# Define schema & table names
source_schema = "refined"
source_table_name = "fact_hourly_weather"
destination_schema = "curated"
destination_table_name = "fact_weather_incremental"
audit_table = "log.fact_weather_audit_log"
summary_table = "log.fact_weather_run_summary"

# -----------------------------------------------
# 🔧 Step A: Get Notebook & Workspace Context
# -----------------------------------------------

try:
    context = notebookutils.runtime.context
    notebook_name  = context.get("currentNotebookName")
    workspace_name = context.get("currentWorkspaceName")
    lakehouse_name = context.get("defaultLakehouseName")
    lakehouse_id   = context.get("defaultLakehouseId")
    workspace_id   = context.get("currentWorkspaceId")
    user_name      = context.get("userName")
    
    print("📌 [ENV INFO]")
    print(f"📓 Notebook   : {notebook_name}")
    print(f"🏢 Workspace  : {workspace_name}")
    print(f"💾 Lakehouse  : {lakehouse_name} ({lakehouse_id})")
    print(f"👤 User       : {user_name}")

except Exception as e:
    notebook_name, lakehouse_id, workspace_id, user_name = "unknown", None, None, "unknown"
    print(f"❌ Failed to fetch notebook context: {str(e)}")
    error_count += 1

# -----------------------------------------------
# 📁 Step B: Build ABFS Paths
# -----------------------------------------------

# Create ABFS path for any table in any schema
def table_path(schema, table):
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{schema}/{table}"

source_path = table_path(source_schema, source_table_name)
destination_path = table_path(destination_schema, destination_table_name)

# -----------------------------------------------
# 📝 Step C: Audit Helpers
# -----------------------------------------------

def log_audit_event(run_id, step, table, message, row_count=0, error_message=""):
    """Writes a detailed event log to audit table"""
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("timestamp", TimestampType(), True),
        StructField("notebook_name", StringType(), True),
        StructField("step", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("row_count", IntegerType(), True),
        StructField("message", StringType(), True),
        StructField("error_message", StringType(), True)
    ])
    data = [{
        "run_id": run_id,
        "timestamp": datetime.utcnow(),
        "notebook_name": notebook_name,
        "step": step,
        "table_name": table,
        "row_count": row_count,
        "message": message,
        "error_message": error_message
    }]
    spark.createDataFrame(data, schema).write.format("delta").mode("append").saveAsTable(audit_table)
    print(f"🧾 [{step}] {message}")

def log_run_summary(run_id, start_ts, end_ts, rows_before, rows_inserted, rows_after, status, error_count):
    """Writes a one-row summary log for the run"""
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("duration_secs", IntegerType(), True),
        StructField("notebook_name", StringType(), True),
        StructField("rows_before", IntegerType(), True),
        StructField("rows_inserted", IntegerType(), True),
        StructField("rows_after", IntegerType(), True),
        StructField("status", StringType(), True),
        StructField("error_count", IntegerType(), True)
    ])
    duration = int((end_ts - start_ts).total_seconds())
    data = [{
        "run_id": run_id,
        "start_time": start_ts,
        "end_time": end_ts,
        "duration_secs": duration,
        "notebook_name": notebook_name,
        "rows_before": rows_before,
        "rows_inserted": rows_inserted,
        "rows_after": rows_after,
        "status": status,
        "error_count": error_count
    }]
    spark.createDataFrame(data, schema).write.format("delta").mode("append").saveAsTable(summary_table)
    print(f"✅ [SUMMARY] {status.upper()}: {rows_inserted} rows in {duration}s | Errors={error_count}")

# -----------------------------------------------
# 🔁 Step D: Perform Incremental Load
# -----------------------------------------------

try:
    # 1. Discover existing table and get max run_date
    try:
        df_existing = spark.read.format("delta").load(destination_path)
        rows_before = df_existing.count()
        max_run_date = df_existing.agg(spark_max("run_date")).first()[0]
        print(f"🔍 Found existing data: {rows_before} rows | Max run_date = {max_run_date}")
        log_audit_event(run_id, "discover_target", destination_path, f"Existing rows={rows_before}, max_run_date={max_run_date}", rows_before)
    except Exception as e:
        max_run_date = None
        print(f"🆕 No previous data found — running full load. Reason: {str(e)}")
        log_audit_event(run_id, "discover_target_error", destination_path, "No previous table", error_message=str(e))
        error_count += 1

    # 2. Load and validate source data
    df_source = spark.read.format("delta").load(source_path)
    total_source = df_source.count()
    log_audit_event(run_id, "read_source", source_path, f"Source rows = {total_source}", total_source)
    df_source.printSchema()
    df_source.show(5, truncate=False)

    # 3. Filter for new data based on run_date
    if max_run_date:
        df_new = df_source.filter(col("run_date") > lit(max_run_date))
        log_audit_event(run_id, "filter_incremental", source_path, f"Filtering run_date > {max_run_date}")
    else:
        df_new = df_source
        log_audit_event(run_id, "filter_full", source_path, "Full load - no prior run_date found")

    rows_inserted = df_new.count()
    log_audit_event(run_id, "count_new", source_path, f"New rows to insert: {rows_inserted}", rows_inserted)

    # 4. Write to target (append or overwrite)
    if rows_inserted > 0:
        mode = "append" if rows_before else "overwrite"
        df_new.write.format("delta").mode(mode).save(destination_path)
        rows_after = spark.read.format("delta").load(destination_path).count()
        log_audit_event(run_id, "write_target", destination_path, f"Inserted {rows_inserted} rows | Mode: {mode}", rows_inserted)
    else:
        rows_after = rows_before
        log_audit_event(run_id, "write_skipped", destination_path, "No new rows to insert")

except Exception as err:
    status = "failed"
    error_count += 1
    print(f"💥 [run_error] {str(err)}")
    log_audit_event(run_id, "run_error", destination_path, "Unhandled exception", error_message=str(err))
    raise

finally:
    # 5. Always log summary
    end_time = datetime.utcnow()
    log_run_summary(run_id, start_time, end_time, rows_before, rows_inserted, rows_after, status, error_count)

StatementMeta(, 7d669e2e-ab56-4574-b6e6-10b9967f6921, 9, Finished, Available, Finished)

📌 [ENV INFO]
📓 Notebook   : NB_SCD_and_Incremental_Loading
🏢 Workspace  : Data Engineering - WeatherAnalytics
💾 Lakehouse  : LH_UK_WeatherData (207b6b26-bde0-4718-95f8-a110b0f5ae31)
👤 User       : Prathy Kamasani
🔍 Found existing data: 7200 rows | Max run_date = 2025-06-10
🧾 [discover_target] Existing rows=7200, max_run_date=2025-06-10
🧾 [read_source] Source rows = 1200
root
 |-- weather_date: date (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- run_date: date (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)

+------------+------------------+------+--------------+------------------------------------+----------+--------------------------+
|weather_date|temperature_2m    |city  |country       |run_id                              |run_date  |ingest_ts                 |
+------------+------------------+------+--------------+---------------------

## 📦 SCD Type 2 

In [34]:
# -----------------------------------------------
# 📦 SCD Type 2 Loader with ABFS Paths & Logging
# -----------------------------------------------

from datetime import datetime
from pyspark.sql.functions import col, current_date, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import uuid
import os

# ─────── 🧠 Load Fabric Runtime Context ───────
ctx = notebookutils.runtime.context
notebook_name  = ctx.get("currentNotebookName")
workspace_name = ctx.get("currentWorkspaceName")
lakehouse_name = ctx.get("defaultLakehouseName")
lakehouse_id   = ctx.get("defaultLakehouseId")
workspace_id   = ctx.get("currentWorkspaceId")
user_name      = ctx.get("userName")

# 🖨️ Print out basic runtime info
print("🔧 Runtime Context Loaded")
print(f"📓 Notebook: {notebook_name}")
print(f"🏢 Workspace: {workspace_name}")
print(f"💾 Lakehouse: {lakehouse_name} ({lakehouse_id})")
print(f"👤 User: {user_name}")

# ─────── ⚙️ Config & Paths ───────
run_id        = str(uuid.uuid4())                 # Unique run ID for logging
start_time    = datetime.utcnow()                 # Capture start timestamp
status        = "success"                         # Default run status
error_count   = 0
batch_id      = str(uuid.uuid4())                 # Batch ID for tracking inserts

# Table references
source_schema = "refined"
source_table  = "city_locations"
dim_schema    = "curated"
dim_table     = "dim_city"
audit_table   = "log.fact_weather_audit_log"
summary_table = "log.fact_weather_run_summary"

# Columns
nat_keys = ["city", "country"]  # Natural keys to identify rows uniquely
attrs    = ["lat", "lng", "population_long"]  # Attributes that may change
insert_count = 0

# 🔗 Function to create ABFS paths based on schema and table
def abfs_path(schema, table):
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/{schema}/{table}"

src_path = abfs_path(source_schema, source_table)
dim_path = abfs_path(dim_schema, dim_table)

# ─────── 📋 Logging Functions ───────

# Audit log for step-by-step actions
def log_audit_event(run_id, step, table, message, row_count=0, error_message=""):
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("timestamp", TimestampType(), True),
        StructField("notebook_name", StringType(), True),
        StructField("step", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField("row_count", IntegerType(), True),
        StructField("message", StringType(), True),
        StructField("error_message", StringType(), True)
    ])
    now = datetime.utcnow()
    data = [{
        "run_id": run_id,
        "timestamp": now,
        "notebook_name": notebook_name,
        "step": step,
        "table_name": table,
        "row_count": row_count,
        "message": message,
        "error_message": error_message
    }]
    spark.createDataFrame(data, schema).write.format("delta").mode("append").saveAsTable(audit_table)
    print(f"📝 [{step}] {message}")

# Final run summary logger
def log_run_summary(run_id, start_ts, end_ts, rows_inserted, status, error_count):
    schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("duration_secs", IntegerType(), True),
        StructField("notebook_name", StringType(), True),
        StructField("rows_inserted", IntegerType(), True),
        StructField("status", StringType(), True),
        StructField("error_count", IntegerType(), True)
    ])
    duration = int((end_ts - start_ts).total_seconds())
    data = [{
        "run_id": run_id,
        "start_time": start_ts,
        "end_time": end_ts,
        "duration_secs": duration,
        "notebook_name": notebook_name,
        "rows_inserted": rows_inserted,
        "status": status,
        "error_count": error_count
    }]
    spark.createDataFrame(data, schema).write.format("delta").mode("append").saveAsTable(summary_table)
    print(f"✅ [SUMMARY] Run {status}: inserted {rows_inserted} rows in {duration}s, errors={error_count}")

# ─────── ⚙️ SCD2 Main Logic ───────

try:
    # Step 1️⃣: Load Source
    df_src = (
        spark.read.format("delta").load(src_path)
             .withColumn("load_ts", current_timestamp())
             .withColumn("batch_id", lit(batch_id))
    )
    src_count = df_src.count()
    log_audit_event(run_id, "load_source", source_table, f"Loaded source data with {src_count} rows", src_count)

    # Step 2️⃣: Check if dimension table exists (via filesystem)
    dim_files = notebookutils.fs.ls(dim_path)
    if not dim_files:
        raise FileNotFoundError("Dimension table folder exists but contains no data")

    # Step 3️⃣: Load dimension and filter current rows
    df_dim_current = (
        spark.read.format("delta").load(dim_path)
             .filter("is_current = true")
             .select(*nat_keys, *attrs)
    )
    log_audit_event(run_id, "load_dim", dim_table, "Loaded current dimension")

    # Step 4️⃣: Find records that have changed values (expire old versions)
    conds = [col(f"t.{a}") != col(f"s.{a}") for a in attrs]
    change_condition = conds[0]
    for c in conds[1:]:
        change_condition |= c

    df_expire = (
        df_dim_current.alias("t")
                     .join(df_src.alias("s"), on=nat_keys, how="inner")
                     .where(change_condition)
                     .select(*[f"t.{k}" for k in nat_keys])
                     .dropDuplicates()
    )

    keys_to_expire = df_expire.collect()

    if keys_to_expire:
        # Convert to DataFrame for filtering
        df_keys = spark.createDataFrame(keys_to_expire).distinct()

        # Update old rows by writing expired versions
        df_dim_update = (
            spark.read.format("delta").load(dim_path)
                 .alias("dim")
                 .join(df_keys.alias("keys"), on=nat_keys)
                 .withColumn("effective_to", current_date())
                 .withColumn("is_current", lit(False))
        )

        df_dim_update.write.format("delta").mode("overwrite").option("replaceWhere", "is_current = true").save(dim_path)
        log_audit_event(run_id, "expire", dim_table, f"Expired {df_keys.count()} rows", df_keys.count())
    else:
        log_audit_event(run_id, "expire", dim_table, "No rows to expire")

    # Step 5️⃣: Insert brand new records (left anti join)
    df_insert = df_src.alias("s").join(
        df_dim_current.select(*nat_keys).distinct(),
        on=nat_keys,
        how="left_anti"
    )
    insert_count = df_insert.count()

    if insert_count > 0:
        df_insert = (
            df_insert.withColumn("effective_from", current_date())
                     .withColumn("effective_to", lit("9999-12-31"))
                     .withColumn("is_current", lit(True))
        )
        df_insert.write.format("delta").mode("append").save(dim_path)
        log_audit_event(run_id, "insert_new", dim_table, f"Inserted {insert_count} new rows", insert_count)
    else:
        log_audit_event(run_id, "insert_skipped", dim_table, "No new rows to insert")

except FileNotFoundError:
    # 🧹 Full load fallback if table not present (first run scenario)
    print("⚠️ No dimension table found – performing full first-time load")
    log_audit_event(run_id, "dim_not_found", dim_table, "Dimension not found – full load fallback")

    df_first_load = (
        df_src.withColumn("effective_from", current_date())
              .withColumn("effective_to", lit("9999-12-31"))
              .withColumn("is_current", lit(True))
    )
    df_first_load.write.format("delta").mode("overwrite").save(dim_path)
    insert_count = df_first_load.count()
    log_audit_event(run_id, "first_load", dim_table, f"Inserted {insert_count} initial rows", insert_count)

except Exception as e:
    # ❌ Catch unexpected errors
    error_count += 1
    status = "failed"
    log_audit_event(run_id, "run_error", dim_table, "Unhandled exception", error_message=str(e))
    raise

finally:
    # ✅ Final step: log summary
    end_time = datetime.utcnow()
    log_run_summary(run_id, start_time, end_time, insert_count, status, error_count)

StatementMeta(, 7d669e2e-ab56-4574-b6e6-10b9967f6921, 36, Finished, Available, Finished)

🔧 Runtime Context Loaded
📓 Notebook: NB_SCD_and_Incremental_Loading
🏢 Workspace: Data Engineering - WeatherAnalytics
💾 Lakehouse: LH_UK_WeatherData (207b6b26-bde0-4718-95f8-a110b0f5ae31)
👤 User: Prathy Kamasani
📝 [load_source] Loaded source data with 5 rows
📝 [load_dim] Loaded current dimension
📝 [expire] No rows to expire
📝 [insert_skipped] No new rows to insert
✅ [SUMMARY] Run success: inserted 0 rows in 8s, errors=0
